In [1]:
# -*- coding: utf-8 -*-
"""train_grpo_qwen_merged.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/standing-on-giants/Meta_hackathon/blob/dev%2Fpratham/train_grpo/train_grpo_qwen_merged.ipynb
"""

import os
import sys
import json
import textwrap
import random
import re
import subprocess
from pathlib import Path

# ==============================================================================
# 1. SETUP & INSTALLATION
# ==============================================================================
print("--- Starting Setup & Installation ---")

subprocess.run(
    "pip install -qU --no-cache-dir unsloth unsloth_zoo transformers trl peft accelerate "
    "bitsandbytes datasets pandas openai python-dotenv",
    shell=True, check=True
)

subprocess.run("rm -rf /kaggle/working/unsloth_compiled_cache", shell=True)
subprocess.run("rm -rf /kaggle/working/Meta_hackathon/unsloth_compiled_cache", shell=True)
print('Installation and Cache Clearance complete.')

# ✅ PATCH GOES HERE — after pip install, before anything else
import transformers.configuration_utils as _tcu
_orig_dumps = json.dumps
def _safe_dumps(obj, **kw):
    class SafeEncoder(json.JSONEncoder):
        def default(self, o):
            return None if callable(o) else super().default(o)
    kw['cls'] = SafeEncoder
    return _orig_dumps(obj, **kw)
_tcu.json.dumps = _safe_dumps
print("JSON encoder patched.")

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = _s.get_secret('META_HACKATHON_TOKEN')
    HF_TOKEN = _s.get_secret('HF_TOKEN')   # ← matches your screenshot exactly
except Exception as e:
    print(f"Warning: Could not load secrets: {e}")
    GITHUB_TOKEN = os.getenv('META_HACKATHON_TOKEN', '')
    HF_TOKEN = os.getenv('HF_TOKEN', '')

# Normalize empty strings to None
HF_TOKEN = (HF_TOKEN or '').strip() or None
GITHUB_TOKEN = (GITHUB_TOKEN or '').strip() or None

print(f"HF_TOKEN loaded: {'YES' if HF_TOKEN else 'NO'}")
print(f"GITHUB_TOKEN loaded: {'YES' if GITHUB_TOKEN else 'NO'}")

REPO_DIR = '/kaggle/working/Meta_hackathon'

if not os.path.exists(REPO_DIR):
    subprocess.run(
        f"git clone -b dev/pratham https://{GITHUB_TOKEN}@github.com/"
        f"standing-on-giants/Meta_hackathon.git {REPO_DIR}",
        shell=True, check=True
    )
else:
    subprocess.run(
        "git fetch origin && "
        "git checkout -- src/__pycache__/ && "  # discard pycache conflicts
        "git checkout dev/pratham && "
        "git pull origin dev/pratham",
        shell=True, cwd=REPO_DIR
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory set to: {os.getcwd()}')

import torch
import numpy as np

from src.environment import DataPipelineEnv
from src.models import PipelineAction, PipelineObservation

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

# ==============================================================================
# 2. LOADING THE MODEL (8-BIT QUANTIZATION)
# ==============================================================================
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

print(f"--- Loading Base Model {MODEL_NAME} ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
    load_in_8bit=True,
    token=HF_TOKEN,
)
print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters()/1e9:.2f}B')

model = FastLanguageModel.get_peft_model(
    model, r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable Parameters: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)')

# === SMOKE TEST: Verify save works before training ===
import shutil
print("Running Smoke Test for Model Saves...")
try:
    model.save_pretrained('/kaggle/working/_smoke_test_lora')
    model.save_pretrained_merged('/kaggle/working/_smoke_test_merged', tokenizer, save_method='merged_16bit')

    # Verify config.json is valid
    with open('/kaggle/working/_smoke_test_merged/config.json', 'r') as f:
        json.loads(f.read())

    print("Smoke test PASSED — safe to proceed.")
    shutil.rmtree('/kaggle/working/_smoke_test_lora', ignore_errors=True)
    shutil.rmtree('/kaggle/working/_smoke_test_merged', ignore_errors=True)
except Exception as e:
    import traceback
    traceback.print_exc()
    raise RuntimeError(f"Smoke Test FAILED: {e}")

# ==============================================================================
# 3. SUPERVISED FINE-TUNING (SFT) - STAGE 1
# ==============================================================================
SYSTEM_PROMPT = textwrap.dedent('''
You are an expert data engineer diagnosing and fixing broken data pipelines.
You receive pipeline state and must choose ONE action per turn.
WORKFLOW: 1. read_data_sample 2. check_schema/compare_schema 3. Apply fix 4. run_pipeline
RULES: Respond with ONLY a JSON object. Never repeat failing actions. dedup for uniqueness failures.
''').strip()

# Shashank's Gold Actions map
GOLD_ACTIONS = {
    'easy': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_orders', 'n_rows': 20}},
        {'action_type': 'add_data_filter', 'params': {'step_id': 'transform_orders', 'filter_condition': 'user_id IS NOT NULL'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'medium': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_order_items', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_items', 'patch_type': 'dedup', 'column': 'order_item_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_ads_insights', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'coalesce', 'column': 'spend'}},
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_conversions', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'dedup', 'column': 'event_id'}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'strip_prefix', 'column': 'campaign_id'}},
        {'action_type': 'alert_upstream_team', 'params': {'team': 'meta_ads_api_team', 'issue_description': 'N/A values in impressions cannot be parsed'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard2': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_ads_insights', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'coalesce', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'dedup', 'column': 'event_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'handle_drift', 'params': {'strategy': 'resolve_column_rename', 'table': 'raw_ads_insights', 'old_column': 'spend', 'new_column': 'total_spend'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
}

def format_obs(obs, step):
    failed = '\n'.join(f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}' for r in obs.failed_assertions) or '  (none)'
    passed = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag = '\n'.join(f'  {n.step_id}: {n.input_table} -> {n.output_table}' for n in obs.dag_structure)
    hist = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs)
    schema = ''
    if obs.current_schema: schema += '\nSCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff: schema += '\nDIFF: ' + json.dumps(obs.schema_diff)
    sample = ''
    if obs.data_sample: sample = '\nDATA: ' + json.dumps(obs.data_sample[:3], default=str)
    actions = '\n'.join(f'  {a}' for a in obs.actions_taken[-4:]) or '  (none)'
    return f'STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})\nDESCRIPTION: {obs.description}\nPIPELINE PASSED: {obs.pipeline_passed}\nLAST RESULT: {obs.last_action_result}\nDAG:\n{dag}\nFAILING:\n{failed}\nPASSING: {passed}\nHISTORY:\n{hist}\nACTIONS:\n{actions}{sample}{schema}\nRespond with ONE action JSON.'

def collect_gold(task_ids=['easy','medium', 'hard', 'hard2'], n_ep=10):
    pairs = []
    for tid in task_ids:
        gold = GOLD_ACTIONS.get(tid, [])
        if not gold: continue
        for _ in range(n_ep):
            env = DataPipelineEnv(task_id=tid)
            res = env.reset()
            obs = res[0] if isinstance(res, tuple) else res
            for si, ad in enumerate(gold, 1):
                pairs.append((format_obs(obs, si), json.dumps(ad)))
                result = env.step(PipelineAction(**ad))
                obs = result.observation
                if obs.pipeline_passed: break
    return pairs

gold_pairs = collect_gold(n_ep=10)
print(f'Collected {len(gold_pairs)} SFT pairs')

from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

sft_texts = [
    tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM_PROMPT},
         {'role':'user','content':obs},
         {'role':'assistant','content':act}],
        tokenize=False, add_generation_prompt=False)
    for obs, act in gold_pairs
]
sft_ds = Dataset.from_dict({'text': sft_texts})
sft_ds_split = sft_ds.train_test_split(test_size=0.1, seed=42)

SFT_DIR = '/kaggle/working/sft_qwen'

# Extreme VRAM preservation applied to SFT
print('--- Starting SFT ---')
# from transformers import EarlyStoppingCallback

sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=sft_ds_split['train'],
    eval_dataset=sft_ds_split['test'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    args=TrainingArguments(
        average_tokens_across_devices=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        eval_strategy='steps',
        eval_steps=20,
        # load_best_model_at_end=True,
        # metric_for_best_model='eval_loss',
        warmup_ratio=0.1,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5, optim='adamw_8bit',
        weight_decay=0.05, lr_scheduler_type='cosine',
        output_dir=SFT_DIR, save_steps=40, seed=42,
    ),
)
sft_stats = sft_trainer.train()
print(f'SFT done. Loss: {sft_stats.training_loss:.4f}')
model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)

HF_REPO = 'Abhinav-hf/qwen-grpo-sft-trained-16bit'

if HF_TOKEN:
    print(f'Pushing seamlessly compiled SFT 16-bit model to Hub: {HF_REPO}')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'SFT Hub upload complete. GRPO will now commence and overwrite the main weights once finished.')
else:
    print('No HF_TOKEN detected — skipping SFT Hub upload.')

# ==============================================================================
# 4. REINFORCEMENT LEARNING ON ENVIRONMENT (GRPO) - STAGE 2
# ==============================================================================
from src.models import PipelineAction, PipelineObservation

def parse_action(text):
    text = re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()
    if '```' in text:
        text = '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```')).strip()
    start = text.find('{')
    if start == -1: return None
    end = text.rfind('}') + 1
    if end <= start: return None
    try:
        data = json.loads(text[start:end])
        if 'action_type' in data: return PipelineAction(**data)
    except: pass
    return None

RECONSTRUCTION_ACTIONS = [
    PipelineAction(action_type='read_data_sample',
                   params={'table': 'raw_ads_insights', 'n_rows': 20}),
    PipelineAction(action_type='compare_schema',
                   params={'table': 'raw_ads_insights'}),
]

def pipeline_reward_fn(completions, task_id=None, n_prior_actions=None, **kwargs):
    rewards = []

    task_ids = task_id if isinstance(task_id, list) else ['hard'] * len(completions)
    priors   = n_prior_actions if isinstance(n_prior_actions, list) else [0] * len(completions)

    for c, tid, n_prior in zip(completions, task_ids, priors):
        text = c if isinstance(c, str) else c[0].get('content', '')
        action = parse_action(text)

        if action is None:
            # Partial credit for having JSON structure but wrong format
            if text.count('{') >= 1 and text.count('}') >= 1:
                rewards.append(-0.1)   # has braces but unparseable
            else:
                rewards.append(-0.3)   # no JSON structure at all
            continue

        reward = 0.3  # format bonus for valid JSON
        try:
            env = DataPipelineEnv(task_id=tid)
            res = env.reset()
            obs = res[0] if isinstance(res, tuple) else res

            for ra in RECONSTRUCTION_ACTIONS[:n_prior]:
                r = env.step(ra)
                obs = r.observation

            n_passed_before = len(obs.passed_assertions)

            result = env.step(action)
            reward += result.reward or 0.0
            obs_after = result.observation

            n_passed_after = len(obs_after.passed_assertions)
            if n_passed_after > n_passed_before:
                reward += 0.2 * (n_passed_after - n_passed_before)

            if action.action_type == 'compare_schema':
                if obs_after.schema_diff and len(obs_after.schema_diff) > 0:
                    reward += 0.3

        except Exception:
            reward -= 0.2

        rewards.append(float(reward))
    return rewards

from src.tasks import TASKS as _available
grpo_task_ids = ['hard', 'hard2']
assert all(t in _available for t in grpo_task_ids), \
    f"Missing tasks. Available: {list(_available.keys())}"

grpo_prompts = []
for tid in grpo_task_ids:
    for _ in range(20):
        # Step 1 prompt — no prior actions
        env = DataPipelineEnv(task_id=tid)
        res = env.reset()
        obs = res[0] if isinstance(res, tuple) else res

        chat = tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user',   'content': format_obs(obs, 1)}],
            tokenize=False, add_generation_prompt=True)
        grpo_prompts.append({'prompt': chat, 'task_id': tid, 'n_prior_actions': 0})

        # Step 2 prompt — after read_data_sample
        r1 = env.step(RECONSTRUCTION_ACTIONS[0])
        obs = r1.observation
        chat = tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user',   'content': format_obs(obs, 2)}],
            tokenize=False, add_generation_prompt=True)
        grpo_prompts.append({'prompt': chat, 'task_id': tid, 'n_prior_actions': 1})

        # Step 3 prompt — after read_data_sample + compare_schema
        r2 = env.step(RECONSTRUCTION_ACTIONS[1])
        obs = r2.observation
        chat = tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user',   'content': format_obs(obs, 3)}],
            tokenize=False, add_generation_prompt=True)
        grpo_prompts.append({'prompt': chat, 'task_id': tid, 'n_prior_actions': 2})

grpo_ds = Dataset.from_list(grpo_prompts)
print(f'GRPO dataset: {len(grpo_ds)} prompts')

from trl import GRPOConfig, GRPOTrainer

GRPO_DIR = '/kaggle/working/grpo_qwen'
grpo_config = GRPOConfig(
    report_to='none',
    average_tokens_across_devices=False,
    output_dir=GRPO_DIR,
    # VRAM Protections: 4 generations maximum inside the rollouts cache
    num_generations=4,
    max_completion_length=200,
    temperature=0.8,
    # do_sample=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    dataloader_num_workers=2,
    num_train_epochs=2,
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    lr_scheduler_kwargs={},
    # fp16=True,
    # bf16=False,
    # Beta acts as a strongly anchored tether to the SFT training rules
    beta=0.5,
    loss_type='grpo',
    logging_steps=1,
    save_steps=40,
    seed=42,
    max_prompt_length=MAX_SEQ_LENGTH,
    max_grad_norm=0.3,
    warmup_steps=20,
)

from peft import prepare_model_for_kbit_training
# model = prepare_model_for_kbit_training(model)
# model.gradient_checkpointing_enable()

grpo_trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=pipeline_reward_fn,
    args=grpo_config,
    train_dataset=grpo_ds,
)

print(f'GRPO: KL beta={grpo_config.beta}, G={grpo_config.num_generations}')
print('Starting GRPO training...')
grpo_stats = grpo_trainer.train()
print(f'GRPO done. Loss: {grpo_stats.training_loss:.4f}')
model.save_pretrained(GRPO_DIR)
tokenizer.save_pretrained(GRPO_DIR)


# ==============================================================================
# 5. EXPORT / PUSH TO HUGGINGFACE HUB
# ==============================================================================

LOCAL_MERGED_DIR = '/kaggle/working/qwen-merged-16bit'
print(f'Saving completely merged 16-bit GRPO model locally to {LOCAL_MERGED_DIR}...')

model.save_pretrained_merged(
    LOCAL_MERGED_DIR,
    tokenizer,
    save_method='merged_16bit'
)
HF_REPO = 'Abhinav-hf/qwen-grpo-complete-trained-16bit'
if HF_TOKEN:
    print(f'Pushing seamlessly compiled GRPO model to Hub: {HF_REPO}')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'Done! Final GRPO Model available at: https://huggingface.co/{HF_REPO}')
else:
    print('No HF_TOKEN detected — skipping Hub upload. Local save complete.')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Extract reward history from trainer logs
log_history = grpo_trainer.state.log_history
steps    = [l['step']   for l in log_history if 'reward' in l]
rewards  = [l['reward'] for l in log_history if 'reward' in l]
losses   = [l['loss']   for l in log_history if 'loss'   in l]
loss_steps = [l['step'] for l in log_history if 'loss'   in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps, rewards, marker='o', color='green')
ax1.set_title('GRPO Mean Episode Reward')
ax1.set_xlabel('Step'); ax1.set_ylabel('Reward')
ax1.grid(True)

ax2.plot(loss_steps, losses, marker='o', color='blue')
ax2.set_title('GRPO Training Loss')
ax2.set_xlabel('Step'); ax2.set_ylabel('Loss')
ax2.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/grpo_training_curves.png', dpi=150)
print('Saved: /kaggle/working/grpo_training_curves.png')

--- Starting Setup & Installation ---
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 197.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 163.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 151.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 193.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 228.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 410.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 188.9 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you ha

Installation and Cache Clearance complete.
JSON encoder patched.
HF_TOKEN loaded: YES
GITHUB_TOKEN loaded: YES


Cloning into '/kaggle/working/Meta_hackathon'...


Working directory set to: /kaggle/working/Meta_hackathon
PyTorch 2.10.0+cu128, CUDA: True
GPU: Tesla T4, VRAM: 15.6GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


...


Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:11<01:11, 71.55s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:00<00:00, 60.14s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Meta_hackathon/Abhinav-hf/qwen-grpo-complete-trained-16bit`
Done! Final GRPO Model available at: https://huggingface.co/Abhinav-hf/qwen-grpo-complete-trained-16bit
Saved: /kaggle/working/grpo_training_curves.png


In [2]:
from peft import PeftModel

LORA_DIR = "/kaggle/working/lora_adapters"

print("Saving LoRA adapters only...")
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

HF_LORA_REPO = "Abhinav-hf/qwen-grpo-lora-adapters"

if HF_TOKEN:
    print(f"Pushing LoRA adapters to Hub: {HF_LORA_REPO}")
    model.push_to_hub(HF_LORA_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_LORA_REPO, token=HF_TOKEN)
    print(f"Done! LoRA adapters available at: https://huggingface.co/{HF_LORA_REPO}")
else:
    print("No HF_TOKEN detected — skipping Hub upload.")

Saving LoRA adapters only...


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_adapters/tokenizer_config.json.


Pushing LoRA adapters to Hub: Abhinav-hf/qwen-grpo-lora-adapters


README.md:   0%|          | 0.00/546 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Saved model to https://huggingface.co/Abhinav-hf/qwen-grpo-lora-adapters


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp6dikefqp/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done! LoRA adapters available at: https://huggingface.co/Abhinav-hf/qwen-grpo-lora-adapters
